# 选择向量模型

向量模型同时处理文档片段和用户问题。选型时不能只看公开排行榜，还要看项目语言、资料是否可以离开本地、最长输入、存储量和自己问题集上的结果。

下面会介绍 MTEB、BERT 类模型、大模型提供的向量调用方式，以及本地和在线模型的差别。价格和榜单名次容易变化，因此这里只说明怎样选择，不给出固定推荐。本页也不比较某组 PDF 问题的改动前后。

## 先看硬条件

1. 资料不能离开本地：只比较可以本地运行的模型。
2. 主要是中文资料：先排除中文效果明显不足的模型。
3. 片段很长：检查超出最长输入后是否会被截断。
4. 资料量大：估算向量维度对存储和检索速度的影响。

MTEB 等公开评测适合用来初选，不能替代项目问题集。BERT 类向量模型通常体积较小；使用大模型作为向量模型时，可能支持更长输入，但运行和存储成本也更高。

In [1]:
candidates = [
    {"name": "本地中文小模型", "chinese": True, "local": True, "max_tokens": 512, "dimensions": 512},
    {"name": "本地多语言模型", "chinese": True, "local": True, "max_tokens": 8192, "dimensions": 1024},
    {"name": "在线向量服务", "chinese": True, "local": False, "max_tokens": 8192, "dimensions": 1536},
]

requirements = {"chinese": True, "must_run_locally": True, "needed_tokens": 400}
remaining = [
    item for item in candidates
    if item["chinese"]
    and (item["local"] or not requirements["must_run_locally"])
    and item["max_tokens"] >= requirements["needed_tokens"]
]

print("硬条件筛选后：", [item["name"] for item in remaining])
print("下一步：用项目问题集比较这些候选，不根据名称直接定案。")

硬条件筛选后： ['本地中文小模型', '本地多语言模型']
下一步：用项目问题集比较这些候选，不根据名称直接定案。


## 怎样比较模型

对每个候选模型使用同一份文档分块、同一批问题、同一个返回数量，至少记录：

- 需要的资料有多少进入前几条；
- 正确资料首次出现在第几条；
- 建立向量库需要多少时间和存储；
- 单次查询的时间；
- 资料是否可以按安全要求处理。

本教程的可下载向量库使用 `BAAI/bge-small-zh-v1.5`，只表示教程实验固定了一个可复现起点，不表示它永远是项目的最佳选择。

## 从公开榜单到项目问题

MTEB（Massive Text Embedding Benchmark）是一组用于比较文本向量模型的任务，包含语义相似、分类、聚类、重排和检索；中文子集常称为 C-MTEB。榜单可以帮助了解模型覆盖的任务和大致能力，但不能直接代替项目测试，因为资料领域、问题写法、分块方式和语言比例都可能不同。榜单分数会变化，这里只说明怎样阅读。

## 两类常见模型

基于 BERT/SBERT（Sentence-BERT，专门把句子编码成向量的架构）的模型以单句编码为主，参数量和运行成本通常较易控制。传统 BERT 直接比较句对时，需要重复处理大量句子；SBERT 先分别编码句子，再计算向量相似度，更适合大规模检索。BGE、E5 等模型也采用这一思路。下面的示意图解释“分别编码再比较”的结构。

![BERT 编码示意](figures/BERT.png)

![SBERT 编码示意](figures/SBERT.png)

另一类是以 decoder-only LLM 为骨干的向量表示模型。它们可能有更长的输入窗口，能在较长文本中保留更多关系，但参数和显存要求也更高；长窗口并不能替代合理分块，因为噪声、索引成本和延迟仍然存在。

## 选型时需要逐项核对

- **语言和领域**：中文、英文、代码或混合资料是否与模型训练分布接近；项目术语是否在自己的问题集上能命中。
- **部署约束**：资料能否发送到外部 API，是否需要离线运行；本地模型还要估算显存、CPU 延迟和并发。
- **输入长度**：模型的 max_tokens 是硬限制还是会截断。分块长度应在同一 tokenizer 下检查。
- **向量维度和存储**：向量维度越大，单条存储和相似度计算通常越贵；维度小也可能损失区分细节。
- **索引和更新**：资料规模、增删频率、批量建立时间和向量库是否支持目标过滤。
- **实际问题集**：用同一批分块、问题、返回数量和距离度量比较目标首次排名、必要页面覆盖、延迟与存储，不要只看平均榜单分数。

模型选定后，文档和问题必须使用同一个向量表示模型与预处理方式；更换模型通常需要重新建立向量索引。


## 本地向量表示模型的代码写法

下面只展示本地向量表示模型的调用写法，不产生本页的选型结论；运行前要确认模型已下载，并在同一批问题上重新比较。

```python
# 本地开源模型的最小调用示例；模型名称、设备和缓存路径应按项目条件修改。
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-zh-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
vector = embedding.embed_query("原来大模型应用开发是如此简单！")
print("向量维度：", len(vector))
print("前五个值：", vector[:5])

# API 向量表示服务也要记录模型版本、维度、输入上限和调用成本。
# 评估时不要在不同候选模型之间混用已经由旧模型生成的向量。
```


## 读 MTEB 页面时看哪些字段

阅读榜单时，要把字段与自己的限制对应起来。`Model Size` 影响运行时间和显存；`Memory Usage` 是特定精度下的估计，不等于所有设备上的实际占用；`Embedding Dimensions` 决定每条向量的存储大小和距离计算量；`Max Tokens` 决定一个片段能否完整编码。`Overall` 汇总了多种任务，检索项目还要单独看 `Text Retrieval` 或中文检索任务，不能用分类成绩代替检索成绩。记录结果时写下查看日期和模型版本。

## 开源和闭源的取舍

本地开源模型通常没有按请求计费和外部调用次数限制，适合资料不能离开本地、调用量稳定或需要定制训练的项目，但要自己承担下载、显存、升级和服务并发。闭源 API 可以把算力和模型维护交给服务商，适合本地资源不足或需要快速验证的阶段，但要核对隐私、价格、速率限制、输入上限和版本变更。两者都应在同一组项目问题上评估。

一个简单的选型表可以这样读：

| 约束 | 检查条件 | 不能省略的复查 |
|---|---|---|
| 中文内部资料，不能外发 | 本地中文向量表示模型 | 专有名词和权限过滤后的命中 |
| 资料量大、延迟敏感 | 较小维度和本地批量推理 | 索引大小、并发和返回数量召回 |
| 片段可能较长 | 输入上限和 tokenizer | 是否被截断、必要证据是否丢失 |
| 领域问法持续失败 | 先查分块、关键词和改写 | 足量标注后才考虑微调 |

选择结果不是模型的永久排名，而是当前资料、分块、问题集和资源约束下的工程决定。
